In [1]:
import geopandas as gpd
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load the shapefile containing the coastal profiles
profiles = gpd.read_file(os.path.join("data", "profiles",  "perfiles_playas_limpiosCuadrantes94a120.shp"))
# Create a new column for the profile ID
profiles["profile_id"] = profiles.index + 1

# Load the shorelines
shorelines = gpd.read_file(os.path.join("data", "shorelines",  "LineasCosta.gpkg"),
                           layer="LineasCostaPlayas")
# Keep only the relevant columns
columns_to_keep = ["ID_LineaCosta", "Municipio", "Playa", "Fecha", "Hora", "Cuadrante", "NivelTotal"]
shorelines = shorelines[columns_to_keep + ["geometry"]]
# Filter the shorelines to keep only those where "Cuadrante" is between 94 and 120
shorelines = shorelines[(shorelines["Cuadrante"] >= 94) & (shorelines["Cuadrante"] <= 120)]

# Ensure both GeoDataFrames use the same coordinate reference system (CRS)
profiles = profiles.to_crs(shorelines.crs)
print(f"Profiles CRS: {profiles.crs}")

Profiles CRS: EPSG:25830


In [3]:
# Intersect profiles and shorelines, retaining only point intersections.
profiles_shoreline_intersections = gpd.overlay(
    profiles[["profile_id", "geometry"]],
    shorelines,
    how="intersection",
    keep_geom_type=False,
).explode(index_parts=False, ignore_index=True)
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections.geometry.geom_type == "Point"
].copy()

# Shapely's project() returns the along-profile distance in the CRS units (meters here).
profile_geometries = profiles.set_index("profile_id").geometry
profiles_shoreline_intersections["shoreline_position_m"] = (
    profiles_shoreline_intersections.apply(
        lambda row: profile_geometries.loc[row["profile_id"]].project(row.geometry),
        axis=1,
    )
)

profiles_shoreline_intersections = gpd.GeoDataFrame(
    profiles_shoreline_intersections[
        ["profile_id"] + columns_to_keep + ["shoreline_position_m", "geometry"]
    ],
    geometry="geometry",
    crs=profiles.crs,
)

# Create a single column for the date and time of the shoreline measurement
profiles_shoreline_intersections["datetime"] = pd.to_datetime(
    profiles_shoreline_intersections["Fecha"].astype(str)
    + " "
    + profiles_shoreline_intersections["Hora"].astype(str),
    format="%Y%m%d %H%M",
    errors="coerce",
)
# Add the same column to the shorelines GeoDataFrame
shorelines["datetime"] = pd.to_datetime(
    shorelines["Fecha"].astype(str) + " " + shorelines["Hora"].astype(str),
    format="%Y%m%d %H%M",
    errors="coerce",
)

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import plotly.express as px
import folium
import pandas as pd

profiles_shoreline_intersections["clean_date"] = pd.to_datetime(
    profiles_shoreline_intersections["Fecha"].astype(str).str[:8],
    format="%Y%m%d",
    errors="coerce"
)

municipalities_list = sorted(profiles_shoreline_intersections["Municipio"].dropna().unique())

dropdown_municipality = widgets.Dropdown(options=municipalities_list, description="Municipality:")
dropdown_beach = widgets.Dropdown(description="Beach:")
select_profiles = widgets.SelectMultiple(description="Profiles:", tooltip="Use Ctrl/Cmd to select multiple")

min_level = float(profiles_shoreline_intersections["NivelTotal"].min())
max_level = float(profiles_shoreline_intersections["NivelTotal"].max())

slider_sea_level = widgets.FloatRangeSlider(
    value=[min_level, max_level],
    min=min_level,
    max=max_level,
    step=0.1,
    description="Sea Level (m):",
    layout=widgets.Layout(width='50%')
)

min_date_val = profiles_shoreline_intersections["clean_date"].min().date()
max_date_val = profiles_shoreline_intersections["clean_date"].max().date()

date_start = widgets.DatePicker(description="Start Date:", value=min_date_val)
date_end = widgets.DatePicker(description="End Date:", value=max_date_val)

button_update = widgets.Button(description="Render Dashboard", button_style="primary")

output_plot = widgets.Output()
output_map = widgets.Output()

def update_beach_options(*args):
    """Populates the beach dropdown based on the active municipality."""
    selected_municipality = dropdown_municipality.value
    beaches = sorted(
        profiles_shoreline_intersections[
            profiles_shoreline_intersections["Municipio"] == selected_municipality
        ]["Playa"].dropna().unique()
    )
    dropdown_beach.options = beaches
    if beaches:
        dropdown_beach.value = beaches[0]
        update_profile_options()

def update_profile_options(*args):
    """Populates the profile selection widget based on the active beach."""
    selected_municipality = dropdown_municipality.value
    selected_beach = dropdown_beach.value
    
    valid_profiles = sorted(
        profiles_shoreline_intersections[
            (profiles_shoreline_intersections["Municipio"] == selected_municipality) &
            (profiles_shoreline_intersections["Playa"] == selected_beach)
        ]["profile_id"].dropna().unique()
    )
    select_profiles.options = valid_profiles
    select_profiles.value = tuple() 

def render_dashboard(b):
    """Executes data filtering, spatial projection, and renders visualizations."""
    municipality = dropdown_municipality.value
    beach = dropdown_beach.value
    level_min, level_max = slider_sea_level.value
    selected_profiles = select_profiles.value
    start_date = pd.to_datetime(date_start.value)
    end_date = pd.to_datetime(date_end.value)
    
    valid_profile_ids = profiles_shoreline_intersections[
        (profiles_shoreline_intersections["Municipio"] == municipality) &
        (profiles_shoreline_intersections["Playa"] == beach)
    ]["profile_id"].unique()
    
    active_profiles = selected_profiles if selected_profiles else valid_profile_ids
    
    df_beach = profiles_shoreline_intersections[
        (profiles_shoreline_intersections["Municipio"] == municipality) &
        (profiles_shoreline_intersections["Playa"] == beach) &
        (profiles_shoreline_intersections["clean_date"].notna()) &
        (profiles_shoreline_intersections["clean_date"] >= start_date) &
        (profiles_shoreline_intersections["clean_date"] <= end_date) &
        (profiles_shoreline_intersections["profile_id"].isin(active_profiles))
    ].copy()
    
    df_beach = df_beach.sort_values(by="clean_date")

    profiles_filt = profiles[profiles["profile_id"].isin(valid_profile_ids)].copy()
    
    shorelines_filt = shorelines[
        (shorelines["Municipio"] == municipality) &
        (shorelines["Playa"] == beach) &
        (shorelines["NivelTotal"] >= level_min) &
        (shorelines["NivelTotal"] <= level_max)
    ].copy()

    profiles_wgs84 = profiles_filt.to_crs(epsg=4326)
    shorelines_wgs84 = shorelines_filt.to_crs(epsg=4326)

    with output_plot:
        clear_output(wait=True)
        if not df_beach.empty:
            fig = go.Figure()
            colors = px.colors.qualitative.Plotly
            
            for idx, pid in enumerate(active_profiles):
                profile_data = df_beach[df_beach["profile_id"] == pid]
                
                in_range = profile_data[
                    (profile_data["NivelTotal"] >= level_min) & 
                    (profile_data["NivelTotal"] <= level_max)
                ]
                
                out_range = profile_data[
                    (profile_data["NivelTotal"] < level_min) | 
                    (profile_data["NivelTotal"] > level_max)
                ]
                
                marker_color = colors[idx % len(colors)]
                
                if not in_range.empty:
                    fig.add_trace(go.Scatter(
                        x=in_range["clean_date"],
                        y=in_range["shoreline_position_m"],
                        mode="markers",
                        marker=dict(color=marker_color, size=7, line=dict(width=0.5, color="white")),
                        name=f"Profile {pid}"
                    ))
                    
                if not out_range.empty:
                    fig.add_trace(go.Scatter(
                        x=out_range["clean_date"],
                        y=out_range["shoreline_position_m"],
                        mode="markers",
                        marker=dict(color="grey", size=5, opacity=0.25),
                        showlegend=False,
                        hoverinfo="skip"
                    ))
                    
            fig.update_layout(
                title=f"Shoreline Evolution - {beach}",
                xaxis_title="Date",
                yaxis_title="Cross-shore Position (m)",
                hovermode="closest",
                template="plotly_white"
            )
            
            fig.show()
        else:
            print("No data available for the selected parameters.")

    with output_map:
        clear_output(wait=True)
        if not profiles_wgs84.empty:
            bounds = profiles_wgs84.total_bounds
            center_lat = (bounds[1] + bounds[3]) / 2
            center_lon = (bounds[0] + bounds[2]) / 2
            
            m = folium.Map(location=[center_lat, center_lon], zoom_start=15)
            
            folium.TileLayer(
                tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                attr="Esri",
                name="Esri Satellite",
                overlay=False,
                control=True
            ).add_to(m)
            
            for _, row in profiles_wgs84.iterrows():
                is_selected = (row["profile_id"] in selected_profiles) if selected_profiles else False
                line_color = "red" if is_selected else "white"
                line_weight = 4 if is_selected else 2
                line_opacity = 1.0 if is_selected else 0.7
                
                geo_json_feature = folium.GeoJson(
                    row.geometry,
                    style_function=lambda x, color=line_color, weight=line_weight, opacity=line_opacity, selected=is_selected: {
                        "color": color, 
                        "weight": weight, 
                        "opacity": opacity,
                        "dashArray": None if selected else "5, 5"
                    }
                )
                
                folium.Tooltip(f"Profile ID: {row['profile_id']}").add_to(geo_json_feature)
                geo_json_feature.add_to(m)
                
            for _, row in shorelines_wgs84.iterrows():
                folium.GeoJson(
                    row.geometry,
                    style_function=lambda x: {"color": "cyan", "weight": 1.5, "opacity": 0.6}
                ).add_to(m)
                
            display(m)
        else:
            print("No spatial data available to render the map.")

dropdown_municipality.observe(update_beach_options, "value")
dropdown_beach.observe(update_profile_options, "value")
button_update.on_click(render_dashboard)

update_beach_options()
update_profile_options()

controls = widgets.VBox([
    widgets.HBox([dropdown_municipality, dropdown_beach, select_profiles]),
    widgets.HBox([date_start, date_end]),
    slider_sea_level,
    button_update
])
display(controls)
display(widgets.VBox([output_plot, output_map]))